# Tiny GPT: masked self-attention from first principles

Goal: inspect vocabulary encoding, the causal mask, forward shape, loss, and random-weight generation. This is an **untrained** model demo, not production-grade language modeling.

From the repo root run `python -m pip install -e ".[dev]"`, then open this notebook.

In [ ]:
import torch
from minigpt import CharTokenizer, ModelConfig, TinyGPT

tokenizer = CharTokenizer.from_text("attention is all you need.\n")
config = ModelConfig(vocab_size=tokenizer.size, block_size=16, n_embed=24, n_layer=2, n_head=4)
torch.manual_seed(9)
model = TinyGPT(config).eval()
print("Vocabulary size:", tokenizer.size)
print("Vocabulary:", "".join(tokenizer.chars))

## Step 1: the mask

`True` means a query token may attend to that key token. Future positions must remain `False`. The same triangular structure applies independently to every attention head.

In [ ]:
mask = model.blocks[0].attn.mask[:6, :6]
print(mask.to(torch.int))
assert not bool(mask[0, 1]) and bool(mask[5, 0])

## Step 2: shape and next-token loss

We predict each following character from the previous context. Inputs and targets are the same shape but shifted by one element.

In [ ]:
encoded = tokenizer.encode("attention is all")
tokens = torch.tensor([encoded[:-1]])
targets = torch.tensor([encoded[1:]])
with torch.no_grad():
    logits, loss = model(tokens, targets)
print("logits:", tuple(logits.shape), "loss:", round(loss.item(), 4))

## Step 3: untrained sampling

Random weights produce meaningless output. Train on enough clean data and evaluate on held-out text before making model-performance claims.

In [ ]:
with torch.no_grad():
    start = torch.tensor([[tokenizer.stoi["a"]]])
    sample = model.generate(start, max_new_tokens=12, temperature=1.0)
print(repr(tokenizer.decode(sample[0].tolist())))